# Downloading Images and Metadata with Python

## Identifying Items/Images
Browse or search the [Library of Congress Digital Collections](https://www.loc.gov/collections/) website to see what is available and refine your search parameters. The website lets you search over 500 curated collections by keyword, format, topic, and division. While [the API documentation](https://www.loc.gov/apis/json-and-yaml/) has detailed information on parameters you can use in your search, searching the collections often gives you good context that is useful in understanding the context of an image within its collection.

Once you've found a search that targets the items that interest you, copy the URL. That will be the base URL for your API request.

The function below will call the API, adding the following parameters:

    fo=json to get JSON format in response
    c=100 a count of 100 results in each response, rather than the default 25
    at=results,pagination provide only the results and pagination parts of the response. The API response is otherwise very long with information we don't need.

Each of the results in the results field will have a field called `image_url`, which has an array of image files related to the item.  

In [1]:
import requests

In [2]:
# Determine that the items exist, and that we can access them
def get_image_urls(url, items=[]):
    '''
    Retrieves the image URLs for items that have public URLs available. 
    Skips over items that are for the colletion as a whole or web pages about the collection.
    Handles pagination. 

    Args: 
        url (str): The URL to request a collection.
        items (list, optional): The list that fetched item URLs will get added to.

    Returns:
        list: The item URLS from the collection.  
    '''
    
    # TODO: call API and store items in results variable
    jsonParams = {"fo":"json", "c":100, "at":"results,pagination"}
    call = requests.get(url, params=jsonParams)
    data = call.json()
    results = data["results"]

    #TODO: Store image URLS in items
    for result in results:
        if "collection" not in result['original_format'] and "web page" not in result['original_format']:
            if result['image_url']:
                item = result['image_url'][-1]
                items.append(item)


    #TODO: make sure we haven't hit the end of the pages, avoiding rate limits
    if data["pagination"]["next"] is not None: 
        #find the URL of the next page
        next_url = data["pagination"]["next"]
        print(f"getting next page: {next_url}")
        #put a limit to how often we call the next page
        time.sleep(3)
        #run this function again
        get_image_urls(next_url, items) 
    return items

In [3]:
images = get_image_urls("https://www.loc.gov/collections/baseball-cards/?dates=1888")

## Downloading Images
The code below will create empty files named after the item's unique identifier. The code will then "write" the image data from the image urls into those empty files, thereby downloading them onto your computer. 

In [5]:
# Access image URLS through API and download them onto our "local" machine
import os
from urllib.parse import urlparse

def get_and_save_images(results_url, path):
    '''
    Takes as input the url for the collection or results set
    e.g. https://www.loc.gov/collections/baseball-cards
    and a list of items (used for pagination)

    Args:
        results_url (str): The url for the collection.
        path (str): The path in which images will be saved.
    '''
    #TODO: call API and store items in results variable
    jsonParams = {"fo":"json", "c":100, "at":"results,pagination"}
    call = requests.get(results_url, params=jsonParams)
    data = call.json()
    results = data["results"]

    #TODO: Find last image in image_url, and store in variable
    for result in results:
        if "collection" not in result['original_format'] and "web page" not in result['original_format']:
            if result['image_url']:
                image = result['image_url'][-1]
                #TODO: create a filename with the identifier portion of the item URL, and save as path
                identifier = urlparse(result['id'])[2].rstrip('/')
                identifier = identifier.split('/')[-1]

                filename = f"{identifier}.jpg"
                filename = os.path.join(path, filename)
            
                #TODO: store image response and write to path
                image_response = requests.get(image, stream=True)

                with open(filename, 'wb') as fd:
                    for chunk in image_response.iter_content(chunk_size=100000):
                        fd.write(chunk)
    
    #TODO: make sure we haven't hit the end of the pages, avoiding rate limits
    if data["pagination"]["next"] is not None: 
        next_url = data["pagination"]["next"]
        print(f"getting next page: {next_url}")
        time.sleep(3)
        get_and_save_images(next_url, path)

In [6]:
os.mkdir("images-named")

In [7]:
get_and_save_images("https://www.loc.gov/collections/baseball-cards/?dates=1888", "images-named")

## Downloading Metadata
To download the collection's metadata, we will need to call our API and store the response as a CSV. One method of accomplishing this is through using the Pandas library, where we can convert the response into a DataFrame, which can more easily be converted into a CSV.

In [11]:
# Use libraries to call API, download into JSON format, and convert into CSV.
import pandas as pd
import requests

def save_metadata(results_url, path):
    #TODO: Call API and store response as JSON
    params = {"fo": "json", "c": "100", "at": "results,pagination"}
    call = requests.get(results_url, params=params)
    data = call.json()
    results = data['results']
    
    #TODO: Convert results to dataframe, use for loop to iterate and add to dataframe. 
    df = pd.DataFrame()
    for result in results:
        if "collection" not in result.get("original_format") and "web page" not in result.get("original_format"):
            df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
    
    #TODO: Convert dataframe to CSV.
    df.to_csv(path + '/metadata.csv', index=False)
    
    #TODO: check if there are other pages
    if data["pagination"]["next"] is not None: # make sure we haven't hit the end of the pages
        next_url = data["pagination"]["next"]
        print(f"getting next page: {next_url}")
        time.sleep(3) # timer to avoid API rate limits
        save_metadata(next_url, path)

In [9]:
os.mkdir("metadata")

In [12]:
save_metadata("https://www.loc.gov/collections/baseball-cards/?dates=1888", "metadata")

## Extra Practice
- How would you add a column with the names of your downloaded image files in your metadata CSV?
- How would you download the metadata of a single item rather than the whole collection?
- How would you select specific metadata fields from each item to be downloaded?